In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import random
import gymnasium as gym
import matplotlib.pyplot as plt
from network import *
from collections import deque
from pathlib import Path
DISTURBANCE_PROBABILITY = 0.01
DISTURBANCE_BURST_LENGTH = 5


In [ ]:
def train_disturbance():

    env = gym.make("CartPole-v1")

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy_net = QNetwork(state_dim, action_dim).to(device)
    target_net = QNetwork(state_dim, action_dim).to(device)

    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(
        policy_net.parameters(),
        lr=LEARNING_RATE
    )

    replay_buffer = ReplayBuffer(BUFFER_SIZE)

    episode_rewards = []
    losses = []
    total_steps = 0

    for episode in range(NUM_EPISODES):

        length = np.random.uniform(
            MIN_POLE_LENGTH, MAX_POLE_LENGTH
        )

        env.unwrapped.length = length

        state, info = env.reset()
        episode_reward = 0
        burst_steps_remaining = 0

        for step in range(MAX_STEPS):

            epsilon = get_epsilon(total_steps)

            if (
                burst_steps_remaining == 0
                and random.random() < DISTURBANCE_PROBABILITY
            ):
                burst_action = random.randrange(action_dim)
                burst_steps_remaining = DISTURBANCE_BURST_LENGTH

            if burst_steps_remaining > 0:
                action = burst_action
                burst_steps_remaining -= 1
            else:
                action = select_action(
                    state, policy_net, epsilon
                )

            next_state, reward, terminated, truncated, info = (
                env.step(action)
            )

            done = terminated or truncated

            replay_buffer.push(
                state,
                action,
                reward,
                next_state,
                terminated
            )

            loss = update_dqn(
                policy_net,
                target_net,
                optimizer,
                replay_buffer
            )

            if loss is not None:
                losses.append(loss)

            state = next_state
            episode_reward += reward
            total_steps += 1

            if total_steps % TARGET_UPDATE_FREQ == 0:
                target_net.load_state_dict(
                    policy_net.state_dict()
                )

            if done:
                break

        episode_rewards.append(episode_reward)

        if (episode + 1) % 100 == 0:
            avg_reward = np.mean(episode_rewards[-100:])

            print(
                f"Episode {episode + 1}/{NUM_EPISODES} | "
                f"Avg reward: {avg_reward:.2f} | "
                f"Epsilon: {epsilon:.3f}"
            )

    env.close()

    return policy_net, episode_rewards, losses

In [ ]:
disturbance_model, rewards, losses = train_disturbance()

In [ ]:
plt.figure(figsize=(10, 5))

moving_average = np.convolve(
    rewards,
    np.ones(100) / 100,
    mode="valid"
)

plt.plot(
    np.arange(99, len(rewards)),
    moving_average,
    label="100-episode moving average"
)

plt.xlabel("Episode")
plt.ylabel("Average reward")
plt.title("Disturbance-Injected DQN Training Performance")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
Path("weights").mkdir(exist_ok=True)

torch.save(
    disturbance_model.cpu().state_dict(),
    "weights/disturbance.pth"
)

print("Saved weights/disturbance.pth")